# 01 — Data Exploration

**Goal**: Connect to Google Earth Engine, pull Landsat imagery for Lanzarote,
and produce visualisations that confirm the pipeline is working.

By the end of this notebook you will have:
- True-colour and false-colour composites for several years
- NDVI maps showing vegetation change across decades
- A basic multi-year comparison to confirm detectable change

**Phase**: 1 — Data acquisition & exploration  
**Data source**: Landsat Collection 2, Level-2 Surface Reflectance (USGS/NASA via GEE)  
**Study area**: Lanzarote + La Graciosa, Canary Islands, Spain


## 0. Setup

Before running, make sure:
1. Your virtual environment is active (`venv\Scripts\Activate.ps1`)
2. You have run `earthengine authenticate` at least once
3. Replace `GEE_PROJECT` below with your actual Cloud project ID


In [ ]:
import sys
import os

# Add project root to path so we can import pipeline modules
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import ee
import folium
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

print('Libraries loaded OK')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Replace with your actual Google Cloud project ID
GEE_PROJECT = 'your-gee-project-id'

# Lanzarote bounding box (WGS-84)
AOI_COORDS = [-13.92, 28.80, -13.30, 29.30]  # [west, south, east, north]

# Map centre for folium
MAP_CENTRE = [29.05, -13.61]
MAP_ZOOM   = 10

# Years to explore (feel free to add more)
EXPLORE_YEARS = [1990, 2000, 2010, 2020, 2023]

In [ ]:
# ── Initialise GEE ─────────────────────────────────────────────────────────────
ee.Initialize(project=GEE_PROJECT)
print('Earth Engine initialised')

# Define AOI geometry
aoi = ee.Geometry.Rectangle(AOI_COORDS)
print(f'AOI defined: {AOI_COORDS}')

## 1. Helper Functions

Cloud masking and compositing functions. These mirror what lives in
`pipeline/acquire.py` — the notebook version is intentionally verbose
so you can see every step.


In [ ]:
# ── Folium helper: add a GEE layer to a folium map ────────────────────────────
def add_ee_layer(folium_map, ee_image, vis_params, name, shown=True, opacity=0.8):
    """Add a GEE image as a tile layer to a folium map."""
    map_id_dict = ee.Image(ee_image).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Map data © Google Earth Engine / USGS',
        name=name,
        overlay=True,
        control=True,
        show=shown,
        opacity=opacity,
    ).add_to(folium_map)


# ── Cloud masking (Landsat Collection 2 QA_PIXEL) ─────────────────────────────
def mask_clouds(image):
    """Mask cloud, cloud shadow, and snow pixels."""
    qa = image.select('QA_PIXEL')
    cloud_mask  = qa.bitwiseAnd(1 << 3).eq(0)   # bit 3 = cloud
    shadow_mask = qa.bitwiseAnd(1 << 4).eq(0)   # bit 4 = cloud shadow
    return image.updateMask(cloud_mask.And(shadow_mask))


# ── Scale Landsat Collection 2 surface reflectance ───────────────────────────
def scale_l8_l9(image):
    """Scale Landsat 8/9 (OLI) surface reflectance to [0, 1]."""
    optical = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    return image.addBands(optical, overwrite=True)

def scale_l5_l7(image):
    """Scale Landsat 5/7 (TM/ETM+) surface reflectance to [0, 1]."""
    optical = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    return image.addBands(optical, overwrite=True)


print('Helper functions defined')

In [ ]:
# ── Load and composite a single year ─────────────────────────────────────────
def get_composite(year):
    """
    Returns a dry-season (May–Sep) median composite for the given year.
    Automatically selects the best available Landsat sensor.
    Bands are renamed to: blue, green, red, nir, swir1, swir2.
    """
    start = f'{year}-05-01'
    end   = f'{year}-09-30'

    if year >= 2022:
        collection_id = 'LANDSAT/LC09/C02/T1_L2'
        bands_in  = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
        scale_fn  = scale_l8_l9
    elif year >= 2013:
        collection_id = 'LANDSAT/LC08/C02/T1_L2'
        bands_in  = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
        scale_fn  = scale_l8_l9
    elif year >= 1999:
        collection_id = 'LANDSAT/LE07/C02/T1_L2'
        bands_in  = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']
        scale_fn  = scale_l5_l7
    else:
        collection_id = 'LANDSAT/LT05/C02/T1_L2'
        bands_in  = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']
        scale_fn  = scale_l5_l7

    bands_out = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']

    collection = (
        ee.ImageCollection(collection_id)
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUD_COVER', 20))
        .map(mask_clouds)
        .map(scale_fn)
        .select(bands_in, bands_out)
    )

    composite = collection.median().clip(aoi)
    n_scenes  = collection.size().getInfo()
    print(f'{year}: {n_scenes} scenes  |  sensor: {collection_id.split("/")[1]}')
    return composite


print('get_composite() ready')

## 2. Check Scene Availability

Before compositing, confirm there are enough cloud-free scenes per year.


In [ ]:
print('Checking scene availability for explore years...\n')
for yr in EXPLORE_YEARS:
    get_composite(yr)   # scene count is printed inside get_composite

## 3. True-Colour Visualisation

Load composites for two years and display them on an interactive map.
Toggle layers using the layer control (top-right corner of the map).


In [ ]:
# Visualisation parameters
true_colour_vis = {
    'bands': ['red', 'green', 'blue'],
    'min': 0.0,
    'max': 0.3,
    'gamma': 1.4,
}

false_colour_vis = {
    'bands': ['nir', 'red', 'green'],   # vegetation shows bright red
    'min': 0.0,
    'max': 0.5,
}

# Build composites
composite_1990 = get_composite(1990)
composite_2023 = get_composite(2023)

# Build folium map
m = folium.Map(location=MAP_CENTRE, zoom_start=MAP_ZOOM,
               tiles='CartoDB dark_matter')

add_ee_layer(m, composite_1990, true_colour_vis,  'True colour — 1990', shown=False)
add_ee_layer(m, composite_2023, true_colour_vis,  'True colour — 2023')
add_ee_layer(m, composite_1990, false_colour_vis, 'False colour (NIR) — 1990', shown=False)
add_ee_layer(m, composite_2023, false_colour_vis, 'False colour (NIR) — 2023', shown=False)

folium.LayerControl(collapsed=False).add_to(m)
display(m)

## 4. NDVI — Vegetation Index

NDVI = (NIR - Red) / (NIR + Red)

Expected values on Lanzarote:
- **Volcanic malpais / barren rock**: near 0 (dark basalt absorbs all bands)
- **Urban areas (Arrecife, Puerto del Carmen)**: -0.1 to 0.1
- **La Geria vineyard zone**: subtle positive signal (~0.1–0.2)
- **Famara cliffs / northern slopes**: slightly higher (~0.15–0.3) after winter rains
- **Ocean**: strong negative (< -0.3)


In [ ]:
ndvi_vis = {
    'min': -0.2,
    'max':  0.5,
    'palette': ['#d73027', '#f46d43', '#fdae61', '#fee08b',
                '#ffffbf', '#d9ef8b', '#a6d96a', '#66bd63', '#1a9850'],
}

def add_ndvi(image):
    return image.normalizedDifference(['nir', 'red']).rename('ndvi')

ndvi_1990 = add_ndvi(composite_1990)
ndvi_2023 = add_ndvi(composite_2023)

m2 = folium.Map(location=MAP_CENTRE, zoom_start=MAP_ZOOM,
                tiles='CartoDB dark_matter')

add_ee_layer(m2, ndvi_1990, ndvi_vis, 'NDVI — 1990', shown=False)
add_ee_layer(m2, ndvi_2023, ndvi_vis, 'NDVI — 2023')

folium.LayerControl(collapsed=False).add_to(m2)
display(m2)

## 5. NDVI Change Map (1990 → 2023)

Positive values (green) = vegetation gain.  
Negative values (red) = vegetation loss.  

We expect to see vegetation loss where coastal resorts expanded.


In [ ]:
ndvi_change = ndvi_2023.subtract(ndvi_1990).rename('ndvi_change')

change_vis = {
    'min': -0.3,
    'max':  0.3,
    'palette': ['#d73027', '#f7f7f7', '#1a9850'],  # red → white → green
}

m3 = folium.Map(location=MAP_CENTRE, zoom_start=MAP_ZOOM,
                tiles='CartoDB dark_matter')

add_ee_layer(m3, ndvi_change, change_vis, 'NDVI Change 1990→2023')
folium.LayerControl(collapsed=False).add_to(m3)
display(m3)

## 6. Multi-Year NDVI Time Series

Sample the mean NDVI across the whole island for each year.
This gives a broad signal of vegetation cover trends.


In [ ]:
print('Computing mean NDVI for each year (this queries GEE — may take ~30s)...\n')

years = list(range(1990, 2024, 2))   # every 2 years for speed in exploration
mean_ndvi_values = []

for yr in years:
    try:
        comp = get_composite(yr)
        ndvi_img = add_ndvi(comp)
        mean_val = ndvi_img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=aoi,
            scale=300,          # coarser scale for speed; use 30 for final analysis
            maxPixels=1e8,
        ).getInfo()['ndvi']
        mean_ndvi_values.append(mean_val)
        print(f'  {yr}: mean NDVI = {mean_val:.4f}')
    except Exception as e:
        mean_ndvi_values.append(None)
        print(f'  {yr}: ERROR — {e}')

# Filter out None values for plotting
valid_pairs = [(y, v) for y, v in zip(years, mean_ndvi_values) if v is not None]
plot_years, plot_values = zip(*valid_pairs)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(plot_years, plot_values, marker='o', linewidth=2,
        color='#2D8C3C', markerfacecolor='white', markeredgewidth=2)
ax.fill_between(plot_years, plot_values, alpha=0.15, color='#2D8C3C')
ax.set_title('Mean NDVI — Lanzarote dry season (May–Sep)', fontsize=14, pad=12)
ax.set_xlabel('Year')
ax.set_ylabel('Mean NDVI')
ax.set_xlim(min(plot_years) - 1, max(plot_years) + 1)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('../data/processed/ndvi_timeseries_lanzarote.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to data/processed/')

## 7. Sanity Checks

Cross-reference against known ground truth before we proceed to classification.


In [ ]:
# ── Spot checks ────────────────────────────────────────────────────────────────
# Known locations and their expected NDVI ranges
spot_checks = {
    'Arrecife city centre':    (28.9631, -13.5495),   # urban → expect NDVI ≈ 0.0
    'Timanfaya lava fields':   (29.0031, -13.7557),   # volcanic → expect NDVI ≈ 0.0
    'La Geria (vineyards)':    (29.0250, -13.6700),   # sparse crops → expect NDVI 0.1–0.2
    'Famara cliffs (scrub)':   (29.1800, -13.5700),   # shrubland → expect NDVI 0.15–0.3
    'Atlantic Ocean':          (28.8500, -13.7000),   # water → expect NDVI < -0.1
}

print('NDVI spot checks — 2023 composite:\n')
for location, (lat, lon) in spot_checks.items():
    point = ee.Geometry.Point([lon, lat])
    val = ndvi_2023.sample(point, 30).first().get('ndvi').getInfo()
    print(f'  {location:35s}  NDVI = {val:.4f}')

## 8. Next Steps

If the spot checks above look reasonable, you're ready for:

**Notebook 02** — Calculate all spectral indices (NDVI, NDWI, NDBI, SAVI, BSI, EVI)
and visualise how each one highlights a different land cover type.

**Notebook 03** — Download CORINE Land Cover labels, remap to our 6 classes,
sample training pixels, and train the Random Forest classifier.

---
*Study area: Lanzarote + La Graciosa, Canary Islands, Spain*  
*Data: Landsat Collection 2, Level-2 Surface Reflectance (USGS/NASA)*  
*CRS: EPSG:32628 (WGS 84 / UTM zone 28N)*
